In [1]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-04 13:19:49--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.175.183.125, 3.175.183.220, 3.175.183.169, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.175.183.125|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  1.17MB/s    in 66s     

2026-03-04 13:20:55 (1.03 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("homework") \
    .getOrCreate()

print(f"Q1 - Spark version: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/05 11:40:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Q1 - Spark version: 4.1.1


In [2]:
from pyspark.sql import functions as F
import os

df = spark.read.parquet("yellow_tripdata_2025-11.parquet")
print(f"Rows: {df.count():,}  |  Columns: {len(df.columns)}")

df.repartition(4).write.parquet("data/yellow/2025/11/", mode="overwrite")

# Check file sizes
files = [f for f in os.listdir("data/yellow/2025/11/") if f.endswith(".parquet")]
sizes = [os.path.getsize(f"data/yellow/2025/11/{f}") / (1024*1024) for f in files]

for f, s in zip(files, sizes):
    print(f"{f}  →  {s:.1f} MB")

print(f"\nQ2 - Average parquet file size: {sum(sizes)/len(sizes):.1f} MB")

Rows: 4,181,444  |  Columns: 20


part-00002-9dcfb3c0-9cdd-461a-8a6d-6edcef4b9236-c000.snappy.parquet  →  24.4 MB
part-00001-9dcfb3c0-9cdd-461a-8a6d-6edcef4b9236-c000.snappy.parquet  →  24.4 MB
part-00000-9dcfb3c0-9cdd-461a-8a6d-6edcef4b9236-c000.snappy.parquet  →  24.4 MB
part-00003-9dcfb3c0-9cdd-461a-8a6d-6edcef4b9236-c000.snappy.parquet  →  24.4 MB

Q2 - Average parquet file size: 24.4 MB


In [3]:
df2 = spark.read.parquet("data/yellow/2025/11/")

q3 = df2.filter(
    F.to_date("tpep_pickup_datetime") == "2025-11-15"
).count()

print(f"Q3 - Trips on Nov 15: {q3:,}")

Q3 - Trips on Nov 15: 162,604


In [4]:
df_duration = df2.withColumn(
    "duration_hours",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 3600
)

q4 = df_duration.agg(F.max("duration_hours").alias("max_hours")).collect()[0]["max_hours"]
print(f"Q4 - Longest trip: {q4:.1f} hours")

Q4 - Longest trip: 90.6 hours


In [5]:
print("Q5 - Spark UI runs on port: 4040")
print(f"Check yours at http://localhost:4040")

Q5 - Spark UI runs on port: 4040
Check yours at http://localhost:4040


In [6]:
df_zones = spark.read \
    .option("header", "true") \
    .csv("taxi_zone_lookup.csv")

df_joined = df2.join(
    F.broadcast(df_zones),
    df2.PULocationID == df_zones.LocationID
)

q6 = df_joined.groupBy("Zone") \
    .count() \
    .orderBy(F.col("count").asc())

q6.show(10)
print(f"Q6 - Least frequent zone: {q6.first()['Zone']}")

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|Governor's Island...|    1|
|Eltingville/Annad...|    1|
|       Arden Heights|    1|
|       Port Richmond|    3|
|       Rikers Island|    4|
| Green-Wood Cemetery|    4|
|         Great Kills|    4|
|   Rossville/Woodrow|    4|
|         Jamaica Bay|    5|
|         Westerleigh|   12|
+--------------------+-----+
only showing top 10 rows


Q6 - Least frequent zone: Governor's Island/Ellis Island/Liberty Island
